# ML Zoomcamp - Module 5: Deploying Machine Learning Models (Homework 5)

This notebook walks through the six questions using the frozen 2026 release
(`pipeline.bin`, `model.py`, `predict.py`, `pyproject.toml`, `uv.lock`, `Dockerfile`, ...).

**Before running:** put this notebook in the same folder as those files and run it with the
project's locked environment (`uv sync --locked`, then start Jupyter with `uv run --with jupyter jupyter lab`),
so that `scikit-learn==1.7.2` is what unpickles the model.

## Setup: verify the model artifact before loading it

`pickle` can execute code on load, so check the file against the published checksum first.

In [ ]:
import hashlib, json, pickle, re, subprocess, sys, time
from pathlib import Path
from urllib.request import Request, urlopen

EXPECTED = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
actual = hashlib.sha256(Path("pipeline.bin").read_bytes()).hexdigest()
print("sha256:", actual)
assert actual == EXPECTED, "pipeline.bin does not match the published checksum - do not load it"
print("checksum OK")

import sklearn
print("scikit-learn in this kernel:", sklearn.__version__)

## Question 1: environment check

Not graded; just record your `uv` version.

In [ ]:
print(subprocess.run(["uv", "--version"], capture_output=True, text=True).stdout.strip())

## Question 2: locked Scikit-Learn version

Read it from both `pyproject.toml` and `uv.lock`.

In [ ]:
print(re.findall(r'scikit-learn[^\n"]*', Path("pyproject.toml").read_text()))

lock = Path("uv.lock").read_text()
m = re.search(r'name = "scikit-learn"\nversion = "([^"]+)"', lock)
print("uv.lock scikit-learn version:", m.group(1))
print("Q2 answer:", m.group(1))

## Question 3: load the model and predict

Uses `normalize_record` from `model.py` so missing-value handling matches training.
Report to 3 decimals (tolerance 0.005).

In [ ]:
from model import normalize_record

with open("pipeline.bin", "rb") as f:
    pipeline = pickle.load(f)

lead_q3 = {
    "lead_source": "paid_ads", "industry": "technology",
    "employment_status": "employed", "location": "north_america",
    "number_of_courses_viewed": 2, "annual_income": 79276.0,
    "interaction_count": 4, "lead_score": 0.41,
}
p3 = float(pipeline.predict_proba([normalize_record(lead_q3)])[0, 1])
print("probability:", p3)
print("Q3 answer:", round(p3, 3))

## Question 4: serve the model

Starts the provided FastAPI app in a background process (same as
`uv run uvicorn predict:app --host 0.0.0.0 --port 9696`), checks `/health`, then posts the second lead.
Stop it afterwards with the last cell of this section.

In [ ]:
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "predict:app", "--host", "127.0.0.1", "--port", "9696"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

def get_json(url, payload=None):
    if payload is None:
        req = Request(url)
    else:
        req = Request(url, data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"}, method="POST")
    with urlopen(req) as r:
        return json.load(r)

for _ in range(30):                      # wait for the server to come up
    try:
        health = get_json("http://127.0.0.1:9696/health"); break
    except Exception:
        time.sleep(0.5)
print(health)

In [ ]:
lead_q4 = {
    "lead_source": "organic_search", "industry": "technology",
    "employment_status": "employed", "location": "europe",
    "number_of_courses_viewed": 4, "annual_income": 80304.0,
    "interaction_count": 7, "lead_score": 0.74,
}
res4 = get_json("http://127.0.0.1:9696/predict", lead_q4)
print(res4)
assert 0 <= res4["conversion_probability"] <= 1 and isinstance(res4["conversion"], bool)
print("Q4 answer:", round(res4["conversion_probability"], 3))

In [ ]:
server.terminate(); server.wait(timeout=10)
print("server stopped")

## Question 5: Dockerfile base image

Read the `FROM` line (the `@sha256:...` digest is not part of the tag).

In [ ]:
first = Path("Dockerfile").read_text().splitlines()[0]
print(first)
tag = first.replace("FROM ", "").split("@")[0]
print("Q5 answer:", tag)

## Question 6: run the container

Needs Docker, so run these in a terminal (they are not run from the notebook):

```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
# in another terminal
python q6_test.py
curl -s http://localhost:9696/health
```

The container runs the same model and code as Q4, so the expected probability is the same.
`/health` must return `"status":"ok"` and the model checksum from `model_metadata.json`.
The cell below checks that expectation without Docker.

In [ ]:
meta = json.load(open("model_metadata.json"))
print("model_sha256 in metadata matches health:", meta["model_sha256"] == health["model_sha256"])
print("Q6 expected answer (same as Q4):", round(res4["conversion_probability"], 3))

## Summary

In [ ]:
print("Q1: (not graded) uv", subprocess.run(["uv", "--version"], capture_output=True, text=True).stdout.strip())
print("Q2:", m.group(1))
print("Q3:", round(p3, 3))
print("Q4:", round(res4["conversion_probability"], 3))
print("Q5:", tag)
print("Q6:", round(res4["conversion_probability"], 3), "(run against the container to confirm)")